# NER-ASR Brain Hack
Przejdziesz przez cały proces: nagranie rozmowy → wgranie audio → transkrypcja ASR → diaryzacja → ocena CER → maskowanie danych wrażliwych → zapis tekstu → porównanie znaczeń zdań.

## Cz. 0 — Przygotuj folder grupy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1yhmhQlDEBv6Xc_hm1dp60VQ5jS63h_QU)

Najpierw zapisz kopię tego notatnika w folderze swojej grupy na [Dysku Google](https://drive.google.com/drive/folders/1CsC9WJdeT7bcbSELHmudqOQV6fTBfkZ4). Następnie umieszczajcie w tym samym folderze nagrania i pozostałe pliki z danymi.

In [ ]:
# instalacja w środowisku notatnika, jeden raz na sesję.
# Osobną instalację opcjonalnej diaryzacji znajdziesz w sekcji 2.
%pip -q install "faster-whisper>=1.2,<2" "jiwer>=4,<5" "transformers>=4.48,<5" "sentence-transformers>=5,<6", "pyannote.audio>=4,<5" sentencepiece numpy pandas matplotlib seaborn

In [ ]:
import re
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio, display

## Cz. 1 — Nagraj rozmowę i wykonaj transkrypcję ASR

W parze nagrajcie 30–60 sekund rozmowy za zgodą uczestników, używając fikcyjnych imion i danych kontaktowych. Mówcie na zmianę i dodajcie krótki fragment z nakładaniem głosów. Zapiszcie nagranie w folderze swojej grupy.

Wgraj jedno nagranie WAV, MP3 lub M4A do Colaba i odsłuchaj je. Nazwa pliku trafi do zmiennej `PLIK_AUDIO`.

In [ ]:
# wgraj jedno nagranie z komputera.
from google.colab import files

przeslane = files.upload()
PLIK_AUDIO = next(iter(przeslane))

In [ ]:
# odsłuchaj wgrane nagranie przed transkrypcją.
display(Audio(filename=PLIK_AUDIO))

### Cz. 1.1 — Transkrypcja faster-whisper

Wybierz rozmiar modelu i język nagrania. Zobacz [opis i porównanie modeli Whisper](https://huggingface.co/openai/whisper-small#model-details) oraz [opis parametru `language`](https://huggingface.co/docs/transformers/model_doc/whisper#transformers.WhisperForConditionalGeneration.generate).

Mniejsze modele działają szybciej, a większe wymagają więcej zasobów. `language` określa język mowy w nagraniu. Wpisz wybrane wartości w dwóch liniach poniżej; warianty do wyboru są w komentarzach.

In [ ]:
# uzupełnij parametry
MODEL_ASR = None  # TODO: wybierz "tiny", "base", "small", "medium" lub "large-v3"
JEZYK_ASR = None  # TODO: wybierz "pl" (polski), "en" (angielski), "de" (niemiecki) lub "fr" (francuski)

segmenty_asr = []
slowa_asr = []
transkrypcja = ""

# Uzupełnij MODEL_ASR i JEZYK_ASR przed uruchomieniem.
from faster_whisper import WhisperModel

model_asr = WhisperModel(MODEL_ASR, device="cpu", compute_type="int8")
generator, informacje = model_asr.transcribe(
    PLIK_AUDIO,
    language=JEZYK_ASR,
    beam_size=5,
    vad_filter=True,
    word_timestamps=True,
)
for segment in generator:
    segmenty_asr.append({
        "start": segment.start, "end": segment.end, "tekst": segment.text.strip()
    })
    for slowo in segment.words or []:
        slowa_asr.append({
            "start": slowo.start, "end": slowo.end, "tekst": slowo.word
        })
transkrypcja = " ".join(s["tekst"] for s in segmenty_asr)
print(transkrypcja)
# Jeśli wynik jest pusty, odsłuchaj nagranie i sprawdź, czy zawiera mowę.
display(pd.DataFrame(segmenty_asr).head(10))
del model_asr  # Dalej potrzebujemy tekstu i czasów, a nie modelu w pamięci.

## Cz. 2 — Przypisz wypowiedzi rozmówcom

Ustaw `CZY_DIARYZACJA = True` i `LICZBA_MOWCOW = 2`, aby rozróżnić rozmówców modelem [pyannote](https://huggingface.co/pyannote/speaker-diarization-community-1). Zaakceptuj warunki modelu i podaj własny [token odczytu](https://huggingface.co/settings/tokens) w ukrytym polu.

Bez dostępu do modelu pomiń diaryzację i przejdź do dalszych części.

In [ ]:
CZY_DIARYZACJA = False
LICZBA_MOWCOW = None

przedzialy_mowcow = []

if CZY_DIARYZACJA:
    from getpass import getpass
    import torch
    from pyannote.audio import Pipeline
    from pyannote.audio.pipelines.utils.hook import ProgressHook
    from faster_whisper.audio import decode_audio

    token_hf = getpass("Token Hugging Face (odczyt): ")
    try:
        diaryzator = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-community-1",
            token=token_hf,
        )
    finally:
        del token_hf

    diaryzator.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    sygnal = decode_audio(PLIK_AUDIO, sampling_rate=16000)
    audio_pamiec = {
        "waveform": torch.from_numpy(sygnal.copy()).unsqueeze(0),
        "sample_rate": 16000,
    }
    parametry = {} if LICZBA_MOWCOW is None else {"num_speakers": LICZBA_MOWCOW}
    with ProgressHook() as postep:
        wynik_diaryzacji = diaryzator(audio_pamiec, hook=postep, **parametry)

    # Wersja exclusive przypisuje w danym momencie najwyżej jednego rozmówcę.
    adnotacja = wynik_diaryzacji.exclusive_speaker_diarization
    for przedzial, _, mowca in adnotacja.itertracks(yield_label=True):
        przedzialy_mowcow.append({
            "start": przedzial.start, "end": przedzial.end, "mowca": mowca
        })
    display(pd.DataFrame(przedzialy_mowcow).head(10))
    del diaryzator, audio_pamiec, sygnal
# Aby wykonać diaryzację, włącz ją w poprzedniej komórce.

### Cz. 2.1 — Połącz słowa i rozmówców

Połącz transkrypcję z etykietami rozmówców na podstawie czasu wypowiedzi. Odsłuchaj miejsca zmiany rozmówcy i sprawdź przypisania, zwłaszcza przy nakładaniu głosów.

In [ ]:
# algorytm niezależny od bibliotek modeli.
def przypisz_mowce(start, end, przedzialy):
    """Przypisuje słowu rozmówcę o najdłuższym wspólnym przedziale czasu.

    Argumenty: start i end — czasy słowa; przedzialy — przedziały rozmówców.
    Zwraca: etykietę rozmówcy albo „NIEZNANY”.
    """
    najlepszy_mowca, najdluzszy_czas = "NIEZNANY", 0.0
    for przedzial in przedzialy:
        wspolny_czas = max(
            0.0, min(end, przedzial["end"]) - max(start, przedzial["start"])
        )
        if wspolny_czas > najdluzszy_czas:
            najlepszy_mowca = przedzial["mowca"]
            najdluzszy_czas = wspolny_czas
    return najlepszy_mowca


def polacz_wypowiedzi(slowa, przedzialy):
    """Łączy kolejne słowa przypisane temu samemu rozmówcy.

    Argumenty: slowa — słowa z czasami; przedzialy — przedziały rozmówców.
    Zwraca: listę wypowiedzi z tekstem, czasami i etykietą rozmówcy.
    """
    wypowiedzi = []
    for slowo in slowa:
        # TODO: wybierz funkcję zdefiniowaną wyżej, która przypisze słowu rozmówcę.
        mowca = przypisz_mowce(slowo["start"], slowo["end"], przedzialy)
        if wypowiedzi and wypowiedzi[-1]["mowca"] == mowca:
            wypowiedzi[-1]["tekst"] += slowo["tekst"]
            wypowiedzi[-1]["end"] = slowo["end"]
        else:
            wypowiedzi.append({**slowo, "mowca": mowca})
    for wypowiedz in wypowiedzi:
        wypowiedz["tekst"] = wypowiedz["tekst"].strip()
    return wypowiedzi


# TODO: wybierz funkcję zdefiniowaną wyżej, która połączy słowa w wypowiedzi.
wypowiedzi = polacz_wypowiedzi(slowa_asr, przedzialy_mowcow)
if wypowiedzi:
    display(pd.DataFrame(wypowiedzi).head(15))
# Wróć tutaj po wykonaniu transkrypcji, aby otrzymać wypowiedzi.

## Cz. 3 — Sprawdź jakość transkrypcji

Przepisz ręcznie oceniany fragment nagrania do `REFERENCJA`. W `HIPOTEZA` pozostaw sam tekst ASR, bez etykiet rozmówców i czasów.

Porównaj CER przed normalizacją i po niej. Niższy CER oznacza mniej błędów znakowych. Znajdź przykładową pomyłkę, np. w nazwisku lub numerze.

In [ ]:
# uzupełnij ręczną referencję.
from jiwer import cer

def normalizuj_do_cer(tekst):
    """Normalizuje tekst przed obliczeniem CER.

    Argumenty: tekst — tekst do normalizacji.
    Zwraca: tekst z ujednoliconą pisownią, interpunkcją i odstępami.
    """
    tekst = unicodedata.normalize("NFC", tekst).lower()
    tekst = "".join(
        " " if unicodedata.category(znak).startswith("P") else znak
        for znak in tekst
    )
    return re.sub(r"\s+", " ", tekst).strip()


# GOTOWY PRZYKŁAD: jedna zamiana w trzech znakach.
print(f"Przykład kot → kod: CER = {cer('kot', 'kod'):.1%}")

REFERENCJA = None  # TODO: własny, ręczny zapis tego samego fragmentu audio.
HIPOTEZA = transkrypcja or None  # Dla fragmentu wpisz odpowiadający mu tekst ASR.
wyniki_cer = None

# Uzupełnij REFERENCJA i sprawdź zgodność fragmentów z HIPOTEZA.
if REFERENCJA is not None and HIPOTEZA is not None:
    # TODO: wybierz funkcję zdefiniowaną wyżej, która normalizuje tekst do CER.
    if not normalizuj_do_cer(REFERENCJA):
        raise ValueError("Referencja po normalizacji jest pusta.")
    wyniki_cer = {
        "CER surowy": cer(REFERENCJA, HIPOTEZA),
        # TODO: wybierz funkcję normalizującą dla obu tekstów przed porównaniem.
        "CER po normalizacji": cer(
            normalizuj_do_cer(REFERENCJA), normalizuj_do_cer(HIPOTEZA)
        ),
    }
    for nazwa, wynik in wyniki_cer.items():
        print(f"{nazwa}: {wynik:.2%}")

## Cz. 4 — Wykryj dane wyrażeniami regularnymi

Uzupełnij słownik `WZORCE` regexami dla telefonu (9 cyfr, opcjonalne `+48`, spacje lub myślniki), e-maila i PESEL-u (11 cyfr). Używaj surowych napisów `r"..."`.

Sprawdź wzorce na przykładach: `+48 123 456 789`, `123-456-789`, `anna.kowalska@example.org` i `12345678901`. Telefon nie powinien dopasować fragmentu PESEL-u. Sprawdź też, co się dzieje, gdy ASR zapisze numer słownie.

In [ ]:
# wybór tekstu roboczego.
TEKST_DEMO = (
    "Anna Kowalska mieszka w Krakowie. Jan Nowak pracuje w Warszawie. "
    "Anna Kowalska podała e-mail anna.kowalska@example.org, "
    "telefon +48 123 456 789 i testowy PESEL 12345678901. "
    "Jan Nowak mówi, że spacer pomaga odpocząć po pracy."
)
tekst_wejsciowy = transkrypcja if transkrypcja.strip() else TEKST_DEMO
print("Źródło:", "własna transkrypcja" if transkrypcja.strip() else "fikcyjny tekst")
print(tekst_wejsciowy)

In [ ]:
# wpisz własne regexy zamiast None.
WZORCE = {
    "TELEFON": None,  # TODO: r"..."
    "EMAIL": None,    # TODO: r"..."
    "PESEL": None,    # TODO: r"..."
}

def wykryj_regex(tekst, wzorce):
    """Wyszukuje w tekście dopasowania podanych wyrażeń regularnych.

    Argumenty: tekst — tekst do sprawdzenia; wzorce — słownik etykiet i regexów.
    Zwraca: listę wykrytych fragmentów z pozycjami i etykietami.
    """
    fragmenty = []
    for etykieta, wzorzec in wzorce.items():
        if wzorzec is None:
            continue
        for trafienie in re.finditer(wzorzec, tekst):
            if trafienie.start() == trafienie.end():
                raise ValueError(f"{etykieta}: wzorzec dopasowuje pusty napis.")
            fragmenty.append({
                "start": trafienie.start(), "end": trafienie.end(),
                "etykieta": etykieta, "wartosc": trafienie.group(),
                "priorytet": 2,
            })
    return fragmenty


# TODO: wybierz funkcję zdefiniowaną wyżej, która wykryje fragmenty regexami.
fragmenty_regex = wykryj_regex(tekst_wejsciowy, WZORCE)
# Uzupełnij trzy wzorce. Pozycje None są pomijane.
display(pd.DataFrame(fragmenty_regex))

### Cz. 4.1 — Sprawdź sumę kontrolną PESEL

Uzupełnij obliczenie cyfry kontrolnej: pomnóż pierwsze 10 cyfr przez wagi `1, 3, 7, 9, 1, 3, 7, 9, 1, 3`, zsumuj wyniki i oblicz `(10 - suma % 10) % 10`. Porównaj wynik z ostatnią cyfrą.

Maskuj również numery z niepoprawną sumą kontrolną.

In [ ]:
# uzupełnij obliczenie cyfry kontrolnej.
def poprawna_suma_pesel(pesel):
    """Sprawdza format PESEL-u i jego cyfrę kontrolną.

    Argumenty: pesel — ciąg 11 cyfr.
    Zwraca: True lub False; None oznacza nieuzupełnione ćwiczenie.
    """
    if re.fullmatch(r"[0-9]{11}", pesel) is None:
        return False
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    cyfra_kontrolna = None  # TODO: oblicz z pierwszych 10 cyfr i wag.
    if cyfra_kontrolna is None:
        return None  # Nieuzupełnione ćwiczenie.
    return cyfra_kontrolna == int(pesel[-1])


for fragment in fragmenty_regex:
    if fragment["etykieta"] == "PESEL":
        # TODO: wybierz funkcję zdefiniowaną wyżej, która sprawdza sumę kontrolną PESEL-u.
        print(fragment["wartosc"], "suma kontrolna:", poprawna_suma_pesel(fragment["wartosc"]))

### Cz. 4.2 — Zamaskuj wykryte fragmenty

Uruchom maskowanie i porównaj tekst przed zmianą i po niej. Sprawdź etykiety telefonu, e-maila i PESEL-u oraz sposób obsługi nakładających się dopasowań.

In [ ]:
# te same funkcje wykorzystasz także do wyników NER.
def wybierz_rozlaczne(fragmenty):
    """Wybiera niepokrywające się fragmenty, uwzględniając priorytet.

    Argumenty: fragmenty — lista wykrytych fragmentów tekstu.
    Zwraca: posortowaną listę wybranych, rozłącznych fragmentów.
    """
    wybrane = []
    kolejnosc = sorted(
        fragmenty,
        key=lambda f: (-f.get("priorytet", 1), f["start"], -(f["end"] - f["start"])),
    )
    for fragment in kolejnosc:
        if not any(
            fragment["start"] < inny["end"] and inny["start"] < fragment["end"]
            for inny in wybrane
        ):
            wybrane.append(fragment)
    return sorted(wybrane, key=lambda f: f["start"])


def maskuj_fragmenty(tekst, fragmenty):
    """Zastępuje wskazane fragmenty tekstu etykietami lub zamiennikami.

    Argumenty: tekst — tekst źródłowy; fragmenty — lista fragmentów do maskowania.
    Zwraca: tekst z zamaskowanymi fragmentami.
    """
    for fragment in sorted(fragmenty, key=lambda f: f["start"], reverse=True):
        start, end = fragment["start"], fragment["end"]
        if not 0 <= start < end <= len(tekst):
            raise ValueError("Pozycje fragmentu nie pasują do tekstu wejściowego.")
        zamiennik = fragment.get("zamiennik", "[" + fragment["etykieta"] + "]")
        tekst = tekst[:start] + zamiennik + tekst[end:]
    return tekst


# TODO: wybierz funkcję zdefiniowaną wyżej, która rozstrzyga nakładające się dopasowania.
wybrane_regex = wybierz_rozlaczne(fragmenty_regex)
# TODO: wybierz funkcję zdefiniowaną wyżej, która maskuje wskazane fragmenty tekstu.
tekst_po_regex = maskuj_fragmenty(tekst_wejsciowy, wybrane_regex)
print(tekst_po_regex)
print(f"Wybrano {len(wybrane_regex)} z {len(fragmenty_regex)} dopasowań.")

## Cz. 5 — Wykryj osoby i miejsca modelem NER

Uruchom model [Babelscape/wikineural-multilingual-ner](https://huggingface.co/Babelscape/wikineural-multilingual-ner) na oryginalnym tekście. Obejrzyj wykryte osoby (`PER`) i miejsca (`LOC`), a następnie połącz je z dopasowaniami regexów. Sprawdź, które nazwy model pominął lub rozpoznał błędnie.

In [ ]:
# model uruchamiamy na CPU.
from transformers import pipeline

MODEL_NER = "Babelscape/wikineural-multilingual-ner"
ner = pipeline(
    "token-classification",
    model=MODEL_NER,
    tokenizer=MODEL_NER,
    aggregation_strategy="simple",
    stride=64,
    device=-1,
)
encje_ner = ner(tekst_wejsciowy)
fragmenty_ner = [
    {
        "start": int(encja["start"]), "end": int(encja["end"]),
        "etykieta": {"PER": "OSOBA", "LOC": "MIEJSCE"}[encja["entity_group"]],
        "wartosc": tekst_wejsciowy[int(encja["start"]):int(encja["end"])],
        "score": float(encja["score"]), "priorytet": 1,
    }
    for encja in encje_ner
    if encja["entity_group"] in {"PER", "LOC"}
]
# TODO: wybierz funkcję zdefiniowaną wyżej, która łączy detektory i usuwa nakładanie.
fragmenty_wspolne = wybierz_rozlaczne(fragmenty_regex + fragmenty_ner)
display(pd.DataFrame(fragmenty_ner))
print(f"Po połączeniu detektorów: {len(fragmenty_wspolne)} fragmentów.")

### Cz. 5.1 — Maskuj bez rozróżniania osób

Uzupełnij `anonimizuj_bez_rozrozniania`, korzystając z `maskuj_fragmenty`. Każdą osobę zastąp etykietą `[OSOBA]`, miejsce etykietą `[MIEJSCE]`, a pozostałe dane etykietami z regexów. Sprawdź wynik ręcznie.

In [ ]:
# napisz ciało funkcji.
def anonimizuj_bez_rozrozniania(tekst, fragmenty):
    """Maskuje wszystkie wykryte dane bez rozróżniania osób.

    Argumenty: tekst — tekst źródłowy; fragmenty — lista wykrytych fragmentów.
    Zwraca: zanonimizowany tekst.
    """
    # TODO: wybierz funkcję maskującą zdefiniowaną wyżej i zwróć wynikowy tekst.
    return None


# TODO: wybierz funkcję zdefiniowaną wyżej, która maskuje dane bez pseudonimów osób.
tekst_bez_rozrozniania = anonimizuj_bez_rozrozniania(
    tekst_wejsciowy, fragmenty_wspolne
)
# Uzupełnij funkcję anonimizuj_bez_rozrozniania.
if tekst_bez_rozrozniania is not None:
    print(tekst_bez_rozrozniania)

### Cz. 5.2 — Nadaj osobom stałe pseudonimy

Uzupełnij `anonimizuj_z_rozroznianiem`: kolejnym osobom nadaj `[OSOBA_1]`, `[OSOBA_2]` itd., zachowując ten sam numer dla powtarzającego się zapisu. Dodaj pole `zamiennik` do kopii fragmentów i użyj `maskuj_fragmenty`.

Różne zapisy tej samej osoby możesz połączyć słownikiem `aliasy` po ręcznym sprawdzeniu tekstu. Miejsca i dane z regexów nadal maskuj etykietami ogólnymi.

In [ ]:
# zaimplementuj stałe oznaczenia osób.
def anonimizuj_z_rozroznianiem(tekst, fragmenty, aliasy=None):
    """Maskuje dane, nadając powtarzającym się osobom stałe pseudonimy.

    Argumenty: tekst — tekst źródłowy; fragmenty — wykryte fragmenty;
        aliasy — opcjonalny słownik łączący różne zapisy tej samej osoby.
    Zwraca: zanonimizowany tekst ze stałymi pseudonimami osób.
    """
    aliasy = {} if aliasy is None else aliasy
    rejestr_osob = {}
    nowe_fragmenty = []
    # TODO:
    # 1. Posortuj fragmenty według początku.
    # 2. Dla OSOBA ustal klucz i nadaj stały numer.
    # 3. Dodaj kopię fragmentu z polem zamiennik do nowe_fragmenty.
    # 4. Wybierz funkcję maskującą zdefiniowaną wyżej i zwróć jej wynik.
    return None


# TODO: wybierz funkcję zdefiniowaną wyżej, która nadaje osobom stałe pseudonimy.
tekst_z_rozroznianiem = anonimizuj_z_rozroznianiem(
    tekst_wejsciowy, fragmenty_wspolne
)
# Uzupełnij funkcję anonimizuj_z_rozroznianiem.
if tekst_z_rozroznianiem is not None:
    print(tekst_z_rozroznianiem)

## Cz. 6 — Zapisz wynik do pliku

Sprawdź zamaskowany tekst i przypisz wybrany wariant do `tekst_do_zapisu`. Zapisz wynik w `wyniki/rozmowa_maskowana.txt`. Aby pobrać plik z Colaba, ustaw `POBIERZ_PLIK = True`.

In [ ]:
# wskaż wynik po własnej ocenie.
tekst_do_zapisu = None  # Wybierz: tekst_bez_rozrozniania albo tekst_z_rozroznianiem.
POBIERZ_PLIK = False

# Przypisz wybrany wynik po maskowaniu do tekst_do_zapisu.
if tekst_do_zapisu is not None:
    if not isinstance(tekst_do_zapisu, str) or not tekst_do_zapisu.strip():
        raise ValueError("Wynik musi być niepustym tekstem.")
    katalog_wynikow = Path("wyniki")
    katalog_wynikow.mkdir(exist_ok=True)
    plik_wynikowy = katalog_wynikow / "rozmowa_maskowana.txt"
    plik_wynikowy.write_text(tekst_do_zapisu.rstrip() + "\n", encoding="utf-8")
    print(f"Zapisano: {plik_wynikowy.resolve()}")
    if POBIERZ_PLIK:
        files.download(str(plik_wynikowy))

## Cz. 7 — Przedstaw wynik

W ciągu 3 minut pokaż transkrypcję i jej CER, porównaj tekst przed maskowaniem i po nim oraz wskaż przykład błędu lub ograniczenia metody. Omów różnice między regexami, NER i wariantami oznaczania osób.

## Cz. 8 — Porównaj znaczenia zdań

Uruchom wielojęzyczny model Sentence Transformer, aby zamienić zdania na embeddingi. Następnie porównaj ich podobieństwo kosinusowe. Wyższy wynik oznacza większe podobieństwo według modelu; nie jest prawdopodobieństwem.

In [ ]:
# instalacja sentence-transformers była w sekcji 0.
from sentence_transformers import SentenceTransformer

MODEL_ZDAN = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_zdan = SentenceTransformer(MODEL_ZDAN, device="cpu")
print("Limit tokenów na tekst:", model_zdan.max_seq_length)

### Cz. 8.1 — Przygotuj zdania i embeddingi

Wpisz do `ZDANIA` od 4 do 8 krótkich zdań: dwie parafrazy, zdanie o innym temacie i parę z zaprzeczeniem. Przewidź najbardziej podobną parę, a następnie oblicz embeddingi i porównaj swoją ocenę z macierzą podobieństwa oraz mapą cieplną.

In [ ]:
# zmień zdania, zachowaj co najmniej dwie niepuste pozycje.
ZDANIA = [
    "" # TODO uzupełnić
]

if len(ZDANIA) < 2 or not all(isinstance(s, str) and s.strip() for s in ZDANIA):
    raise ValueError("Podaj przynajmniej dwa niepuste zdania.")

etykiety_zdan = [f"Z{i + 1}" for i in range(len(ZDANIA))]
embeddingi = model_zdan.encode(
    ZDANIA, normalize_embeddings=True, convert_to_numpy=True
)
macierz_podobienstwa = embeddingi @ embeddingi.T
print("Kształt embeddingów (zdania, wymiary):", embeddingi.shape)
display(pd.DataFrame({"etykieta": etykiety_zdan, "zdanie": ZDANIA}))

In [ ]:
# tabela i mapa cieplna podobieństwa kosinusowego.
tabela_podobienstwa = pd.DataFrame(
    macierz_podobienstwa, index=etykiety_zdan, columns=etykiety_zdan
)
display(tabela_podobienstwa.round(3))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    tabela_podobienstwa,
    annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, square=True,
    cbar_kws={"label": "Podobieństwo kosinusowe"}, ax=ax,
)
ax.set_title("Podobieństwo znaczeń zdań")
ax.set_xlabel("Zdanie")
ax.set_ylabel("Zdanie")
plt.tight_layout()
plt.show()

### Cz. 8.2 — Wyświetl graficznie statystyki

Oblicz macierz korelacji współrzędnych embeddingów przez `pd.DataFrame(embeddingi.T).corr()` i przedstaw ją na mapie cieplnej. Porównaj wynik z podobieństwem kosinusowym. Korelacja współrzędnych nie oznacza zależności między uczestnikami.

In [ ]:
# TODO - wykonać macież korelacji